# RUKOPYS YOLO + Qwen3-VL Submit (VLM-R3 + HALP)


In [ ]:
INSTALL_DEPS = True
INSTALL_DOCLAYOUT_YOLO = True
DOCLAYOUT_REPO = "/kaggle/working/DocLayout-YOLO"

if INSTALL_DEPS:
    import subprocess
    import sys

    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            "accelerate",
            "peft",
            "bitsandbytes",
            "qwen-vl-utils",
            "pandas==2.2.2",
            "pillow<12",
        ],
        [sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/huggingface/transformers.git"],
    ]
    for cmd in commands:
        print("Running:", " ".join(cmd), flush=True)
        subprocess.check_call(cmd)

if INSTALL_DOCLAYOUT_YOLO:
    import subprocess
    import sys
    from pathlib import Path

    repo = Path(DOCLAYOUT_REPO)
    if not repo.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/opendatalab/DocLayout-YOLO.git", str(repo)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo)])


In [ ]:
import gc
import json
import logging
import math
import os
import re
import shutil
import subprocess
import sys
import time
import warnings
from pathlib import Path
from types import ModuleType

import pandas as pd
import torch
from PIL import Image, ImageFilter, ImageStat
from torchvision.ops import nms
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"


def suppress_transformers_noise():
    message = r".*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*"
    warnings.filterwarnings("ignore", message=message)
    logging.getLogger("transformers").setLevel(logging.ERROR)
    logging.getLogger("transformers.processing_utils").setLevel(logging.ERROR)
    try:
        from transformers.utils import logging as hf_logging

        hf_logging.set_verbosity_error()
    except Exception:
        pass


suppress_transformers_noise()

for repo_path in ["/kaggle/working/DocLayout-YOLO", "/kaggle/input/DocLayout-YOLO", "/kaggle/input/doclayout-yolo/DocLayout-YOLO"]:
    p = Path(repo_path)
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Some DocLayout-YOLO checkpoints reference this callback module while loading.
if "doclayout_yolo.utils.callbacks.hub" not in sys.modules:
    dummy_hub = ModuleType("doclayout_yolo.utils.callbacks.hub")
    dummy_hub.callbacks = {}
    sys.modules["doclayout_yolo.utils.callbacks.hub"] = dummy_hub

from doclayout_yolo import YOLOv10

BASE_MODEL_CANDIDATES = [
    "/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1",
    "/kaggle/input/qwen3-vl-8b-instruct",
    "Qwen/Qwen3-VL-8B-Instruct",
]

LORA_CANDIDATES = [
    "/kaggle/working/qwen3vl_rukopys_stage2_gold_vlm_r3_halp/qwen3vl_rukopys_lora_final",
    "/kaggle/input/qwen3vl-rukopys-stage2-vlm-r3-halp/qwen3vl_rukopys_lora_final",
]

YOLO_WEIGHT_CANDIDATES = [
    "/kaggle/input/models/notpitomon/htd-box-doclayoutyolo-v4/pytorch/default/2/DoclayoutYoloV4.1.pt",
    "/kaggle/input/htd-box-doclayoutyolo-v4/DoclayoutYoloV4.1.pt",
    "/kaggle/input/doclayoutyolov4-1/DoclayoutYoloV4.1.pt",
    "/kaggle/input/doclayout-yolo-v4-1/DoclayoutYoloV4.1.pt",
    "/kaggle/working/DoclayoutYoloV4.1.pt",
]

# Expected structure:
#   DATASET_ROOT/test/images/*.jpg
#   DATASET_ROOT/test/metadata.jsonl
DATASET_ROOT = "/kaggle/input/datasets/quii29/rukopys-dataset"

VALIDATION_RECORDS_CANDIDATES = [
    "/kaggle/working/qwen3vl_rukopys_stage2_gold_vlm_r3_halp/gold_validation_records.jsonl",
    "/kaggle/input/qwen3vl-rukopys-stage2-vlm-r3-halp/gold_validation_records.jsonl",
    "/kaggle/working/qwen3vl_rukopys_curriculum/gold_validation_records.jsonl",
    "/kaggle/input/qwen3vl-rukopys-curriculum/gold_validation_records.jsonl",
]

RUN_SPLIT = "test"  # choices: "test", "validation"
OUTPUT_CSV = "submission_vlm_r3_halp.csv"
TEST_MODE = False

# OCR settings. Keep "all_text" for the strongest text pass.
CROP_OCR_MODE = "halp_vlm_r3"  # choices: "none", "smart", "all_text", "halp_vlm_r3"
CROP_BATCH_SIZE = 2
CHECKPOINT_EVERY = 10
PROGRESS_LOG_EVERY = 1  # reliable Kaggle log line after every processed image
PROGRESS_BAR_WIDTH = 20
USE_TQDM_PROGRESS = False  # multiprocessing tqdm is often hidden in Kaggle logs
RESUME_PARTIALS = True
HYBRID_PARTIAL_PREFIX = "vlm_r3_halp_partial_results_gpu"
CHECKPOINT_INPUT_DIR = ""  # optional: set to a Kaggle input folder containing hybrid partial CSVs

# YOLO settings copied from the provided YOLO notebook/checkpoint metadata.
YOLO_IMG_SIZE = 1280
YOLO_CONF = 0.20
YOLO_MAX_DET = 220
YOLO_IOU_NMS = 0.60
YOLO_DEDUP_IOU = 0.90
YOLO_PAD_SCALE_X = 0.00
YOLO_PAD_SCALE_Y = 0.00

MAX_PIXELS_PAGE = 850_000
MAX_PIXELS_CROP = 262_144
MAX_NEW_TOKENS_CROP = 192
CROP_PAD_RATIO = 0.04


# VLM-R3 + HALP routing settings.
ILLEGIBLE_TOKEN = "[illegible]"
USE_QWEN_PAGE_JSON = False  # keep YOLO as the submit detector; enable only for ablation/ensemble tests
QWEN_PAGE_JSON_MAX_NEW_TOKENS = 2048
QWEN_YOLO_MERGE_IOU = 0.45
USE_HALP_PROBE = True
HALP_RISK_THRESHOLD = 0.50
HALP_FORMULA_TABLE_BASE_RISK = 0.45
HALP_LOW_CONF_THRESHOLD = 0.35
HALP_LOW_CONF_BONUS = 0.18
HALP_SMALL_AREA_THRESHOLD = 0.0025
HALP_SMALL_AREA_BONUS = 0.18
HALP_BLUR_THRESHOLD = 12.0
HALP_BLUR_BONUS = 0.12
HALP_EMPTY_TEXT_BONUS = 0.25

VLM_R3_MAX_REFINEMENT_PASSES = 3
VLM_R3_FORCE_TYPES = {"formula", "table"}
VLM_R3_CONTEXT_SCALE = 1.35
VLM_R3_MIN_AREA = 0.125
VLM_R3_MAX_AREA = 0.5
MAX_PIXELS_REFINE = 524_288
MAX_PIXELS_TABLE = 786_432
MAX_NEW_TOKENS_REFINE = 256
LOAD_LORA_CROP_PROMPTS = False  # keep the stronger hybrid prompts below instead of adapter-saved crop prompts

SPECIAL_TEXT_MARKER_RULES = (
    "Use these special markers only when they are visible in the crop: "
    "~~word~~ for strikethrough text, ~~old~~{new} for strikethrough text with a visible correction, "
    "and [illegible] for an unreadable word inside an otherwise legible line. "
)

DOCUMENT_CONTEXT_RULES = (
    "The crop may come from Ukrainian dictation handwriting, historical Ukrainian/Cyrillic documents, "
    "school homework, exams, tables, formulas, chemistry notation, teacher marks, or mixed handwriting/print. "
    "Read only visible characters. Do not complete from canonical or memorized text. "
    "Preserve old spelling and do not modernize. "
)

COMMON_OCR_RULES = (
    "Return only the transcription. No JSON, no Markdown, no explanation. "
    + DOCUMENT_CONTEXT_RULES
    + SPECIAL_TEXT_MARKER_RULES
    + "Preserve punctuation, line content, corrections, spelling mistakes, capitalization, digits, "
    "abbreviations, quotes, hyphens, line-final dashes, and visible spacing as much as possible. "
    "Do not translate, correct grammar, normalize spelling, expand abbreviations, summarize, "
    "or infer hidden/missing text."
)

PAGE_PROMPT = (
    "Extract every visible document region. Return only a compact JSON array. "
    "Each item must have keys bbox,type,text. bbox is [x1,y1,x2,y2] on a 0-1000 grid. "
    "type is one of handwritten,printed,formula,table,annotation,image,graph. "
    "Use empty text for image and graph. Preserve reading order. "
    "For every text field, transcribe only visible characters exactly; do not translate, summarize, "
    "normalize spelling, correct grammar, expand abbreviations, or infer hidden/missing text. "
    "For unreadable visible words use [illegible]. "
    "No Markdown, no explanation."
)

CROP_PROMPTS = {
    "handwritten": (
        "Transcribe the visible handwritten text exactly. "
        "Preserve punctuation, line content, corrections, and strikethrough markers. "
        + COMMON_OCR_RULES
    ),
    "printed": (
        "Transcribe the visible printed or typed text exactly. "
        "Preserve punctuation, line content, corrections, and strikethrough markers. "
        + COMMON_OCR_RULES
    ),
    "annotation": (
        "Read this short annotation, teacher mark, grade, correction, or numbering. "
        "Return only the exact visible text. "
        + SPECIAL_TEXT_MARKER_RULES
        + "No explanation."
    ),
    "formula": (
        "Read this standalone math, logic, vector, matrix, determinant, set/relation, statistics, physics, "
        "or chemistry expression exactly as written. Return only formula text, using LaTeX when it is the "
        "clearest representation and plain Unicode when it better matches the handwriting. Do not wrap the "
        "answer in dollar signs. Preserve visible symbols, indices, superscripts, subscripts, arrows, fractions, "
        "matrix/determinant structure, punctuation, numbering, and strikethrough/correction markers. "
        "Do not solve, simplify, normalize, explain, or convert old notation into a different style. If unreadable, return [illegible]."
    ),
    "table": (
        "Read this table region exactly. Return only pipe-separated table text. Use one output line per visual row "
        "and | between cells. Preserve empty cells with empty fields, for example A||C. Preserve row order, "
        "column order, multi-word cell text, wrapped cell text, numbers, units, punctuation, dashes, visible spelling "
        "mistakes, corrections, and strikethrough markers. Do not infer missing cells, rebalance columns, summarize, "
        "or explain. If unreadable, return [illegible]."
    ),
    "image": "Return an empty string.",
    "graph": "Return an empty string.",
    "default": "Transcribe the visible content exactly. " + COMMON_OCR_RULES,
}

VALID_TYPES = {"handwritten", "printed", "formula", "table", "annotation", "image", "graph"}
TEXT_TYPES = {"handwritten", "printed", "formula", "table", "annotation"}
IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".webp", ".bmp"]


In [ ]:
def find_model_id():
    for item in BASE_MODEL_CANDIDATES:
        if item.startswith("/") and Path(item).exists():
            return item
        if not item.startswith("/"):
            return item
    raise FileNotFoundError("No base model found. Add Qwen3-VL to Kaggle input or enable internet.")


def find_lora_dir():
    for item in LORA_CANDIDATES:
        p = Path(item)
        if (p / "adapter_config.json").exists():
            return p
    searched = chr(10).join(f"  - {p}" for p in LORA_CANDIDATES)
    raise FileNotFoundError(
        "No VLM-R3+HALP Stage 2 LoRA adapter_config.json found. Checked:" + chr(10) + searched
    )


def find_yolo_weights():
    for item in YOLO_WEIGHT_CANDIDATES:
        p = Path(item)
        if p.exists():
            return p
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for p in input_root.rglob("*.pt"):
            name = p.name.lower()
            parent = str(p.parent).lower()
            if "doclayout" in name or "doclayout" in parent or "yolo" in name or "yolo" in parent:
                return p
    raise FileNotFoundError("No DocLayout-YOLO .pt weights found. Add the YOLO weight dataset and update YOLO_WEIGHT_CANDIDATES if needed.")


def get_dataset_root():
    root = Path(DATASET_ROOT)
    required_split = "train" if RUN_SPLIT == "validation" else "test"
    metadata_path = root / required_split / "metadata.jsonl"
    if not metadata_path.exists():
        raise FileNotFoundError(
            f"DATASET_ROOT is not configured correctly: {root}. Expected {required_split}/metadata.jsonl under this path."
        )
    return root


def find_validation_records_path(lora_path):
    candidates = [Path(p) for p in VALIDATION_RECORDS_CANDIDATES]
    candidates.append(Path(lora_path).parent / "gold_validation_records.jsonl")
    candidates.append(Path(lora_path).parent.parent / "gold_validation_records.jsonl")
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("No gold_validation_records.jsonl found. Set RUN_SPLIT='test' or mount the validation records.")


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def resolve_image_path(root, split, file_name):
    raw = Path(file_name)
    name = raw.name
    stem = raw.stem
    candidate_names = [name] + [stem + ext for ext in IMAGE_EXTENSIONS if stem + ext != name]
    candidates = [root / split / file_name, root / file_name]
    for candidate_name in candidate_names:
        candidates.extend([root / split / "images" / candidate_name, root / split / candidate_name])
    for p in candidates:
        if p.exists():
            return str(p)
    return str(root / split / "images" / candidate_names[0])


def load_prompt_config(lora_dir):
    global CROP_PROMPTS, MAX_PIXELS_CROP
    cfg_path = Path(lora_dir) / "rukopys_prompt_config.json"
    if not cfg_path.exists():
        return
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    if LOAD_LORA_CROP_PROMPTS:
        CROP_PROMPTS.update(cfg.get("crop_prompts", {}))
    MAX_PIXELS_CROP = int(cfg.get("max_pixels_crop", MAX_PIXELS_CROP))


model_id = find_model_id()
lora_dir = find_lora_dir()
yolo_weights = find_yolo_weights()
dataset_root = get_dataset_root()
load_prompt_config(lora_dir)

if RUN_SPLIT == "validation":
    validation_path = find_validation_records_path(lora_dir)
    test_records = read_jsonl(validation_path)
    IMAGE_SPLIT = "train"
    OUTPUT_CSV = "vlm_r3_halp_validation_pred.csv"
    print("Validation records:", validation_path)
else:
    test_records = read_jsonl(dataset_root / "test" / "metadata.jsonl")
    IMAGE_SPLIT = "test"
    OUTPUT_CSV = "submission_vlm_r3_halp.csv"

if TEST_MODE:
    test_records = test_records[:4]

print("Base model:", model_id)
print("LoRA:", lora_dir)
print("YOLO weights:", yolo_weights)
print("Dataset:", dataset_root)
print("Run split:", RUN_SPLIT)
print("Images:", len(test_records))


In [ ]:
checkpoint_dir = Path(CHECKPOINT_INPUT_DIR) if CHECKPOINT_INPUT_DIR else None
if checkpoint_dir and checkpoint_dir.exists():
    print("Restoring hybrid checkpoints from", checkpoint_dir)
    for src in checkpoint_dir.glob(f"{HYBRID_PARTIAL_PREFIX}*.csv"):
        dst = Path("/kaggle/working") / src.name
        shutil.copy2(src, dst)
        print(f"Copied {src} -> {dst}")
else:
    print("No hybrid checkpoint input configured.")

print("Current hybrid checkpoint files:")
for p in sorted(Path("/kaggle/working").glob(f"{HYBRID_PARTIAL_PREFIX}*.csv")):
    try:
        rows = len(pd.read_csv(p).drop_duplicates(subset=["image"], keep="last"))
    except Exception:
        rows = "?"
    print(f"{p} size={p.stat().st_size} rows={rows}")


In [ ]:
def normalize_type(value):
    value = str(value or "handwritten").strip().lower()
    return value if value in VALID_TYPES else "handwritten"


def clamp_xyxy(box, width, height):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return None
    try:
        x1, y1, x2, y2 = [float(v) for v in box]
    except Exception:
        return None
    x1, x2 = sorted((max(0, min(width, x1)), max(0, min(width, x2))))
    y1, y2 = sorted((max(0, min(height, y1)), max(0, min(height, y2))))
    if x2 - x1 < 3 or y2 - y1 < 3:
        return None
    return [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]


def iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    return inter / max(1, area_a + area_b - inter)


def sort_regions(regions):
    return sorted(regions, key=lambda r: (r["bbox"][1], r["bbox"][0]))


def strip_internal_fields(region):
    return {"bbox": region["bbox"], "type": normalize_type(region.get("type")), "text": str(region.get("text") or "")}


def clean_crop_text(text):
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    text = re.sub(r"^(text|transcription|answer)\s*:\s*", "", text, flags=re.I).strip()
    if len(text) >= 2 and text[0] == text[-1] and text[0] in {"'", '"'}:
        text = text[1:-1].strip()
    if text.startswith("[") or text.startswith("{"):
        try:
            obj = json.loads(text)
            if isinstance(obj, dict) and "text" in obj:
                text = str(obj["text"])
            else:
                return ""
        except Exception:
            return ""
    return text[:500]


In [ ]:
def get_yolo_names(yolo_model):
    names = getattr(yolo_model, "names", None)
    if names is None and hasattr(yolo_model, "model"):
        names = getattr(yolo_model.model, "names", None)
    return names or {}


def get_class_name(names, cls_id):
    if isinstance(names, dict):
        return names.get(cls_id, names.get(str(cls_id), "handwritten"))
    if isinstance(names, (list, tuple)) and 0 <= cls_id < len(names):
        return names[cls_id]
    return "handwritten"


def load_yolo_model(device):
    yolo_model = YOLOv10(str(yolo_weights))
    try:
        yolo_model.to(device)
    except Exception as e:
        print("YOLO .to(device) skipped:", e, flush=True)
    return yolo_model


def dedupe_yolo_regions(regions):
    kept = []
    for region in sorted(regions, key=lambda r: float(r.get("_score", 0.0)), reverse=True):
        duplicate = any(iou(region["bbox"], old["bbox"]) > YOLO_DEDUP_IOU for old in kept)
        if not duplicate:
            kept.append(region)
    return sort_regions(kept)


def postprocess_yolo_result(result, img_w, img_h, names):
    boxes = []
    scores = []
    labels = []
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        boxes.append([x1, y1, x2, y2])
        scores.append(float(box.conf[0]))
        labels.append(int(box.cls[0]))

    if not boxes:
        return []

    boxes_t = torch.tensor(boxes, dtype=torch.float32)
    scores_t = torch.tensor(scores, dtype=torch.float32)
    labels_t = torch.tensor(labels, dtype=torch.int64)

    keep_indices = []
    for cls_id in sorted(set(labels)):
        cls_mask = labels_t == cls_id
        cls_indices = torch.nonzero(cls_mask, as_tuple=True)[0]
        kept = nms(boxes_t[cls_indices], scores_t[cls_indices], YOLO_IOU_NMS)
        keep_indices.extend(cls_indices[kept].tolist())

    regions = []
    for idx in sorted(set(keep_indices)):
        x1, y1, x2, y2 = boxes_t[idx].tolist()
        h = max(1.0, y2 - y1)
        pad_x = YOLO_PAD_SCALE_X * h
        pad_y = YOLO_PAD_SCALE_Y * h
        box = clamp_xyxy([x1 - pad_x, y1 - pad_y, x2 + pad_x, y2 + pad_y], img_w, img_h)
        if box is None:
            continue
        cls_id = int(labels_t[idx].item())
        rtype = normalize_type(get_class_name(names, cls_id))
        regions.append({"bbox": box, "type": rtype, "text": "", "_score": float(scores_t[idx].item())})

    return dedupe_yolo_regions(regions)


def detect_regions_yolo(yolo_model, image_path, yolo_device):
    results = yolo_model.predict(
        source=str(image_path),
        imgsz=YOLO_IMG_SIZE,
        conf=YOLO_CONF,
        max_det=YOLO_MAX_DET,
        verbose=False,
        device=yolo_device,
    )
    result = results[0]
    if hasattr(result, "orig_shape") and result.orig_shape:
        img_h, img_w = result.orig_shape
    else:
        with Image.open(image_path) as img:
            img_w, img_h = img.size
    return postprocess_yolo_result(result, img_w=img_w, img_h=img_h, names=get_yolo_names(yolo_model))


In [ ]:
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
import torch.nn as nn


class HALPProbe(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        hidden = min(1024, max(128, input_dim // 2))
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


HALP_PROBE = None
HALP_PROBE_META = {}


def configure_processor_for_generation(processor):
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.padding_side = "left"
    return processor


def load_halp_probe(device):
    if not USE_HALP_PROBE:
        return None, {}
    probe_path = Path(lora_dir) / "halp_probe.pt"
    if not probe_path.exists():
        print("HALP probe not found; using heuristic risk routing.", flush=True)
        return None, {}
    try:
        payload = torch.load(probe_path, map_location="cpu")
        probe = HALPProbe(int(payload["input_dim"]))
        probe.load_state_dict(payload["state_dict"])
        probe.eval().to(device)
        meta = {
            "layer": int(payload.get("layer", -1)),
            "threshold": float(payload.get("threshold", 0.5)),
        }
        print("Loaded HALP probe:", probe_path, meta, flush=True)
        return probe, meta
    except Exception as e:
        print("Could not load HALP probe; using heuristic risk routing:", e, flush=True)
        return None, {}


def load_qwen_model(device):
    global HALP_PROBE, HALP_PROBE_META
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    base = AutoModelForImageTextToText.from_pretrained(
        model_id,
        device_map={"": device},
        quantization_config=quantization_config,
        dtype=torch.float16,
        trust_remote_code=True,
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    )
    model = PeftModel.from_pretrained(base, str(lora_dir))
    model.eval()
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    processor = configure_processor_for_generation(processor)
    if processor.tokenizer.pad_token_id is not None:
        model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    HALP_PROBE, HALP_PROBE_META = load_halp_probe(device)
    return model, processor


def apply_chat_template(processor, messages):
    candidates = [
        {"tokenize": False, "add_generation_prompt": True, "template_kwargs": {"enable_thinking": False}},
        {"tokenize": False, "add_generation_prompt": True, "processor_kwargs": {"enable_thinking": False}},
        {"tokenize": False, "add_generation_prompt": True, "enable_thinking": False},
        {"tokenize": False, "add_generation_prompt": True},
    ]
    for kwargs in candidates:
        try:
            with warnings.catch_warnings():
                warnings.filterwarnings(
                    "ignore",
                    message=r".*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*",
                )
                return processor.apply_chat_template(messages, **kwargs)
        except TypeError:
            continue
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def encode_generation_batch(processor, messages_batch, device):
    processor.tokenizer.padding_side = "left"
    texts = [apply_chat_template(processor, m) for m in messages_batch]
    image_inputs, video_inputs = process_vision_info(messages_batch)
    try:
        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            text_kwargs={"padding": True, "return_tensors": "pt"},
            images_kwargs={"return_tensors": "pt"},
            videos_kwargs={"return_tensors": "pt"},
        )
    except TypeError:
        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
    return inputs.to(device)


def generate_batch(model, processor, messages_batch, device, max_new_tokens):
    inputs = encode_generation_batch(processor, messages_batch, device)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
        )
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    decoded = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    del inputs, out, trimmed
    torch.cuda.empty_cache()
    return decoded


def halp_probe_predict(model, processor, messages, device):
    if HALP_PROBE is None:
        return None
    try:
        inputs = encode_generation_batch(processor, [messages], device)
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = model(
                **inputs,
                output_hidden_states=True,
                use_cache=False,
                return_dict=True,
            )
        layer = int(HALP_PROBE_META.get("layer", -1))
        hidden = outputs.hidden_states[layer]
        mask = inputs["attention_mask"]
        last_idx = mask.sum(dim=1).clamp(min=1) - 1
        feat = hidden[torch.arange(hidden.shape[0], device=hidden.device), last_idx].float()
        with torch.no_grad():
            score = torch.sigmoid(HALP_PROBE(feat.to(next(HALP_PROBE.parameters()).device))).item()
        del inputs, outputs, hidden, feat
        torch.cuda.empty_cache()
        return float(score)
    except Exception as e:
        print("HALP probe forward failed; falling back to heuristic:", e, flush=True)
        torch.cuda.empty_cache()
        return None


In [ ]:
def resize_to_pixel_budget(img, max_pixels):
    w, h = img.size
    total = max(1, w * h)
    if total <= max_pixels:
        return img
    scale = (max_pixels / total) ** 0.5
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    return img.resize((new_w, new_h), Image.Resampling.LANCZOS)


def expand_box(bbox, img_w, img_h, scale=1.0, pad_ratio=0.0):
    x1, y1, x2, y2 = [float(v) for v in bbox]
    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    side_pad = max(bw, bh) * pad_ratio
    bw = bw * scale + 2 * side_pad
    bh = bh * scale + 2 * side_pad
    return clamp_xyxy([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], img_w, img_h)


def vlm_r3_zoom_scale(bbox, img_w, img_h):
    area = max(1, (bbox[2] - bbox[0]) * (bbox[3] - bbox[1]))
    ratio = area / max(1, img_w * img_h)
    if ratio < VLM_R3_MIN_AREA:
        return 2.0
    if ratio >= VLM_R3_MAX_AREA:
        return 1.0
    return 2.0 - ((ratio - VLM_R3_MIN_AREA) / (VLM_R3_MAX_AREA - VLM_R3_MIN_AREA))


def crop_image_variant(image_path, bbox, scale=1.0, pad_ratio=CROP_PAD_RATIO, max_pixels=MAX_PIXELS_CROP):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        w, h = img.size
        box = expand_box(bbox, w, h, scale=scale, pad_ratio=pad_ratio)
        if box is None:
            box = bbox
        crop = img.crop(tuple(box))
        if scale > 1.0:
            cw, ch = crop.size
            crop = crop.resize((max(1, int(cw * scale)), max(1, int(ch * scale))), Image.Resampling.LANCZOS)
        return resize_to_pixel_budget(crop, max_pixels)


def crop_image(image_path, bbox):
    return crop_image_variant(image_path, bbox, scale=1.0, pad_ratio=CROP_PAD_RATIO, max_pixels=MAX_PIXELS_CROP)


def box_area_ratio(bbox, img_w, img_h):
    return max(0, bbox[2] - bbox[0]) * max(0, bbox[3] - bbox[1]) / max(1, img_w * img_h)


def blur_score_for_region(image_path, bbox):
    try:
        with Image.open(image_path) as img:
            img = img.convert("L")
            w, h = img.size
            box = expand_box(bbox, w, h, scale=1.0, pad_ratio=CROP_PAD_RATIO)
            crop = img.crop(tuple(box or bbox)).resize((128, 128), Image.Resampling.BILINEAR)
            edges = crop.filter(ImageFilter.FIND_EDGES)
            stat = ImageStat.Stat(edges)
            return float(stat.var[0])
    except Exception:
        return 999.0


def page_messages(image_path):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path, "max_pixels": MAX_PIXELS_PAGE},
                {"type": "text", "text": PAGE_PROMPT},
            ],
        }
    ]


def extract_json_array(text):
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return obj
    except Exception:
        pass
    start = text.find("[")
    end = text.rfind("]")
    if start >= 0 and end > start:
        try:
            obj = json.loads(text[start:end + 1])
            if isinstance(obj, list):
                return obj
        except Exception:
            return []
    return []


def qwen_bbox_to_pixels(bbox, img_w, img_h):
    if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
        return None
    try:
        vals = [float(v) for v in bbox]
    except Exception:
        return None
    # The page_json training prompt uses a 0-1000 grid. If the model already emits pixels,
    # values usually exceed 1000 on full-resolution pages, so leave them as-is.
    if max(vals) <= 1000 and (img_w > 1000 or img_h > 1000):
        vals = [vals[0] * img_w / 1000.0, vals[1] * img_h / 1000.0, vals[2] * img_w / 1000.0, vals[3] * img_h / 1000.0]
    return clamp_xyxy(vals, img_w, img_h)


def parse_qwen_page_regions(text, img_w, img_h):
    regions = []
    for item in extract_json_array(text):
        if not isinstance(item, dict):
            continue
        box = qwen_bbox_to_pixels(item.get("bbox"), img_w, img_h)
        if box is None:
            continue
        rtype = normalize_type(item.get("type"))
        text_value = "" if rtype in {"image", "graph"} else clean_crop_text(item.get("text", ""))
        regions.append({"bbox": box, "type": rtype, "text": text_value, "_source": "qwen_page", "_score": 0.55})
    return sort_regions(regions)


def detect_regions_qwen_page(qwen_model, processor, image_path, device):
    if not USE_QWEN_PAGE_JSON:
        return []
    try:
        with Image.open(image_path) as img:
            img_w, img_h = img.size
        out = generate_batch(qwen_model, processor, [page_messages(image_path)], device, QWEN_PAGE_JSON_MAX_NEW_TOKENS)[0]
        regions = parse_qwen_page_regions(out, img_w, img_h)
        print(f"Qwen page_json regions={len(regions)}", flush=True)
        return regions
    except Exception as e:
        print("Qwen page_json failed; using YOLO only:", e, flush=True)
        torch.cuda.empty_cache()
        return []


def merge_qwen_yolo_regions(qwen_regions, yolo_regions):
    merged = [dict(r, _source=r.get("_source", "yolo")) for r in yolo_regions]
    for q in qwen_regions:
        best_idx, best_iou = None, 0.0
        for idx, old in enumerate(merged):
            overlap = iou(q["bbox"], old["bbox"])
            if overlap > best_iou:
                best_idx, best_iou = idx, overlap
        if best_idx is not None and best_iou >= QWEN_YOLO_MERGE_IOU:
            target = merged[best_idx]
            qtype = normalize_type(q.get("type"))
            if qtype in {"formula", "table"} or normalize_type(target.get("type")) == "handwritten":
                target["type"] = qtype
            if str(q.get("text") or "").strip() and not str(target.get("text") or "").strip():
                target["text"] = str(q.get("text") or "")
            target["_source"] = target.get("_source", "yolo") + "+qwen_page"
            target["_qwen_iou"] = best_iou
        else:
            merged.append(dict(q))
    return dedupe_yolo_regions(merged)


def should_crop_ocr(region):
    if CROP_OCR_MODE == "none":
        return False
    if region.get("type") not in TEXT_TYPES:
        return False
    if CROP_OCR_MODE == "all_text":
        return True
    if CROP_OCR_MODE == "halp_vlm_r3":
        if normalize_type(region.get("type")) in VLM_R3_FORCE_TYPES:
            return True
        text = str(region.get("text") or "").strip()
        if "qwen_page" in str(region.get("_source", "")) and text and len(text) >= 4 and ILLEGIBLE_TOKEN not in text.lower():
            return False
        return True
    text = str(region.get("text") or "")
    return (not text.strip()) or len(text) < 4 or len(text) > 160


def crop_prompt_for_region(region, refinement=False):
    rtype = normalize_type(region.get("type"))
    prompt = CROP_PROMPTS.get(rtype, CROP_PROMPTS["default"])
    if not refinement:
        return prompt
    if rtype == "formula":
        return (
            "You are reading a difficult formula region. Use every provided image view as visual evidence. "
            "Return only the exact visible formula or [illegible]. Do not solve, simplify, or explain. "
            + prompt
        )
    if rtype == "table":
        return (
            "You are reading a difficult table region. Use every provided image view as visual evidence. "
            "Return only pipe-separated table text, one visual row per line, or [illegible]. "
            + prompt
        )
    return "Use the provided crop and zoomed views. Return only the visible transcription or [illegible]. " + prompt


def crop_messages(image_path, region):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": crop_image(image_path, region["bbox"])},
                {"type": "text", "text": crop_prompt_for_region(region, refinement=False)},
            ],
        }
    ]


def vlm_r3_refine_messages(image_path, region, img_w, img_h):
    rtype = normalize_type(region.get("type"))
    bbox = region["bbox"]
    zoom_scale = vlm_r3_zoom_scale(bbox, img_w, img_h)
    max_pixels = MAX_PIXELS_TABLE if rtype == "table" else MAX_PIXELS_REFINE
    normal = crop_image_variant(image_path, bbox, scale=1.0, pad_ratio=CROP_PAD_RATIO, max_pixels=max_pixels)
    zoom = crop_image_variant(image_path, bbox, scale=zoom_scale, pad_ratio=CROP_PAD_RATIO, max_pixels=max_pixels)
    context = crop_image_variant(image_path, bbox, scale=VLM_R3_CONTEXT_SCALE, pad_ratio=CROP_PAD_RATIO, max_pixels=max_pixels)
    action = {"action": "zoom", "bbox_2d": bbox, "reason": f"{rtype} high HALP risk"}
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": normal},
                {"type": "text", "text": "Initial detected region."},
                {"type": "image", "image": zoom},
                {"type": "text", "text": "VLM-R3 zoomed region from action " + json.dumps(action, ensure_ascii=False) + "."},
                {"type": "image", "image": context},
                {"type": "text", "text": "Wider context view. " + crop_prompt_for_region(region, refinement=True)},
            ],
        }
    ]


def halp_heuristic_score(region, image_path, img_w, img_h):
    rtype = normalize_type(region.get("type"))
    score = 0.0
    signals = []
    if rtype in VLM_R3_FORCE_TYPES:
        score += HALP_FORMULA_TABLE_BASE_RISK
        signals.append("formula_table")
    yolo_score = float(region.get("_score", 1.0))
    if yolo_score < HALP_LOW_CONF_THRESHOLD:
        score += HALP_LOW_CONF_BONUS
        signals.append("low_yolo_conf")
    if box_area_ratio(region["bbox"], img_w, img_h) < HALP_SMALL_AREA_THRESHOLD:
        score += HALP_SMALL_AREA_BONUS
        signals.append("small_area")
    blur = blur_score_for_region(image_path, region["bbox"])
    if blur < HALP_BLUR_THRESHOLD:
        score += HALP_BLUR_BONUS
        signals.append("blur")
    text = str(region.get("text") or "").strip()
    if (not text) or ILLEGIBLE_TOKEN in text.lower():
        score += HALP_EMPTY_TEXT_BONUS
        signals.append("empty_or_illegible")
    return min(1.0, score), signals


def halp_region_score(qwen_model, processor, image_path, region, device, img_w, img_h):
    heuristic, signals = halp_heuristic_score(region, image_path, img_w, img_h)
    probe_score = None
    if USE_HALP_PROBE and HALP_PROBE is not None:
        probe_score = halp_probe_predict(qwen_model, processor, crop_messages(image_path, region), device)
    score = max(heuristic, probe_score if probe_score is not None else 0.0)
    region["_halp_risk"] = float(score)
    region["_halp_signals"] = ",".join(signals)
    if probe_score is not None:
        region["_halp_probe"] = float(probe_score)
    return score


def output_is_bad(text):
    text = str(text or "").strip()
    if not text:
        return True
    lowered = text.lower()
    if lowered in {"none", "null", "n/a"}:
        return True
    if lowered.startswith("{") or lowered.startswith("[") and not lowered.startswith(ILLEGIBLE_TOKEN.lower()):
        return True
    if "i cannot" in lowered or "sorry" in lowered:
        return True
    return False


def candidate_quality(text, rtype):
    text = clean_crop_text(text)
    if output_is_bad(text):
        return -1000
    if text == ILLEGIBLE_TOKEN:
        return -10
    score = min(len(text), 120) / 120.0
    if rtype == "formula":
        math_chars = sum(1 for ch in text if ch.isdigit() or ch in "=+-*/^_{}\\()[]<>√∑∫≈≤≥→←×÷")
        score += min(1.5, math_chars / 12.0)
    elif rtype == "table":
        separators = text.count("|") + text.count("\n")
        score += min(1.5, separators / 6.0)
    if ILLEGIBLE_TOKEN in text:
        score -= 0.5
    return score


def choose_best_candidate(candidates, rtype):
    cleaned = [clean_crop_text(c) for c in candidates]
    ranked = sorted(((candidate_quality(c, rtype), c) for c in cleaned), key=lambda x: x[0], reverse=True)
    if not ranked or ranked[0][0] < 0:
        return ILLEGIBLE_TOKEN
    return ranked[0][1] or ILLEGIBLE_TOKEN


def refine_region_vlm_r3(qwen_model, processor, image_path, region, device, img_w, img_h):
    rtype = normalize_type(region.get("type"))
    message_variants = [crop_messages(image_path, region), vlm_r3_refine_messages(image_path, region, img_w, img_h)]
    if rtype == "table":
        table_region = dict(region)
        table_region["type"] = "table"
        message_variants.append(vlm_r3_refine_messages(image_path, table_region, img_w, img_h))
    try:
        outs = generate_batch(qwen_model, processor, message_variants, device, MAX_NEW_TOKENS_REFINE)
    except Exception as e:
        print("VLM-R3 refinement failed:", e, flush=True)
        torch.cuda.empty_cache()
        return ILLEGIBLE_TOKEN
    return choose_best_candidate(outs[:VLM_R3_MAX_REFINEMENT_PASSES], rtype)


def ocr_regions(qwen_model, processor, image_path, regions, device):
    with Image.open(image_path) as img:
        img_w, img_h = img.size
    crop_indices = [i for i, r in enumerate(regions) if should_crop_ocr(r)]
    high_risk_indices, normal_indices = [], []
    for i in crop_indices:
        risk = halp_region_score(qwen_model, processor, image_path, regions[i], device, img_w, img_h)
        forced = normalize_type(regions[i].get("type")) in VLM_R3_FORCE_TYPES
        if risk >= HALP_RISK_THRESHOLD or forced:
            high_risk_indices.append(i)
        else:
            normal_indices.append(i)

    for start in range(0, len(normal_indices), CROP_BATCH_SIZE):
        batch_indices = normal_indices[start:start + CROP_BATCH_SIZE]
        msgs = [crop_messages(image_path, regions[i]) for i in batch_indices]
        try:
            outs = generate_batch(qwen_model, processor, msgs, device, MAX_NEW_TOKENS_CROP)
        except Exception as e:
            print("Crop OCR batch failed:", e, flush=True)
            torch.cuda.empty_cache()
            gc.collect()
            continue
        for idx, out in zip(batch_indices, outs):
            text = clean_crop_text(out)
            if text:
                regions[idx]["text"] = text

    for idx in high_risk_indices:
        text = refine_region_vlm_r3(qwen_model, processor, image_path, regions[idx], device, img_w, img_h)
        regions[idx]["text"] = text if text else ILLEGIBLE_TOKEN

    for region in regions:
        if normalize_type(region.get("type")) in TEXT_TYPES and not str(region.get("text") or "").strip():
            region["text"] = ILLEGIBLE_TOKEN
    return sort_regions([strip_internal_fields(r) for r in regions])


def infer_one_image(qwen_model, processor, yolo_model, image_path, device, yolo_device):
    yolo_regions = detect_regions_yolo(yolo_model, image_path, yolo_device)
    qwen_regions = detect_regions_qwen_page(qwen_model, processor, image_path, device)
    regions = merge_qwen_yolo_regions(qwen_regions, yolo_regions)
    if not regions:
        return []
    regions = ocr_regions(qwen_model, processor, image_path, regions, device)
    return sort_regions(regions)


In [ ]:
import multiprocessing as mp


def format_duration(seconds):
    if seconds is None or seconds <= 0:
        return "--:--"
    seconds = int(round(seconds))
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    if hours:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def progress_bar(done, total, width=PROGRESS_BAR_WIDTH):
    ratio = done / max(1, total)
    filled = min(width, max(0, int(round(width * ratio))))
    return "█" * filled + " " * (width - filled)


def print_progress_line(gpu_id, done, total, run_done, elapsed, region_count, image_name):
    pct = 100.0 * done / max(1, total)
    speed = run_done / elapsed if run_done > 0 and elapsed > 0 else 0.0
    sec_per_img = elapsed / run_done if run_done > 0 else 0.0
    remaining = max(0, total - done)
    eta = remaining / speed if speed > 0 else None
    bar = progress_bar(done, total)
    last = str(image_name or "")[:18]
    if speed > 0:
        timing = f"{format_duration(elapsed)}<{format_duration(eta)}, {sec_per_img:.2f}s/img, speed={speed:.2f} img/s"
    else:
        timing = f"resume, speed=-- img/s"
    print(
        f"GPU {gpu_id}: {pct:3.0f}%|{bar}| {done}/{total} "
        f"[{timing}, regions={region_count}, last={last}]",
        flush=True,
    )


def worker_process(gpu_id, records, output_csv):
    suppress_transformers_noise()
    device = f"cuda:{gpu_id}"
    total_records = len(records)
    print(f"[GPU {gpu_id}] loading Qwen and YOLO for {total_records} images", flush=True)
    qwen_model, processor = load_qwen_model(device)
    yolo_model = load_yolo_model(device)
    print(f"[GPU {gpu_id}] models loaded; starting hybrid inference", flush=True)

    done = set()
    results = []
    if RESUME_PARTIALS and Path(output_csv).exists():
        try:
            old = pd.read_csv(output_csv)
            old = old.drop_duplicates(subset=["image"], keep="last")
            done = set(old["image"].tolist())
            results = old.to_dict("records")
            print(f"[GPU {gpu_id}] resumed {len(done)}/{total_records} rows from {output_csv}", flush=True)
        except Exception as e:
            print(f"[GPU {gpu_id}] could not read checkpoint: {e}", flush=True)

    print_progress_line(gpu_id, len(done), total_records, 0, 0.0, 0, "resumed" if done else "start")

    pbar = None
    if USE_TQDM_PROGRESS:
        pbar = tqdm(
            total=total_records,
            initial=len(done),
            desc=f"GPU {gpu_id}",
            position=gpu_id,
            leave=True,
            dynamic_ncols=True,
            smoothing=0.05,
            unit="img",
            mininterval=1.0,
            maxinterval=10.0,
            file=sys.stdout,
        )
    run_start = time.time()
    run_done = 0

    for rec in records:
        image_name = Path(rec["file_name"]).name
        if image_name in done:
            continue
        image_path = resolve_image_path(dataset_root, IMAGE_SPLIT, rec["file_name"])
        try:
            regions = infer_one_image(qwen_model, processor, yolo_model, image_path, device, gpu_id)
        except Exception as e:
            print(f"[GPU {gpu_id}] failed {image_name}: {e}", flush=True)
            regions = []
            torch.cuda.empty_cache()
            gc.collect()
        results.append({"image": image_name, "regions": json.dumps(regions, ensure_ascii=False)})
        done.add(image_name)
        run_done += 1
        elapsed = max(1e-6, time.time() - run_start)
        if pbar is not None:
            pbar.set_postfix_str(
                f"speed={run_done / elapsed:.2f} img/s, last_regions={len(regions)}, last={image_name[:12]}"
            )
            pbar.update(1)
        if run_done % PROGRESS_LOG_EVERY == 0 or len(done) == total_records:
            print_progress_line(gpu_id, len(done), total_records, run_done, elapsed, len(regions), image_name)

        if len(results) % CHECKPOINT_EVERY == 0:
            tmp = output_csv + ".tmp"
            pd.DataFrame(results).drop_duplicates(subset=["image"], keep="last").to_csv(tmp, index=False)
            os.replace(tmp, output_csv)
            if pbar is not None:
                pbar.set_postfix_str(
                    f"speed={run_done / elapsed:.2f} img/s, last_regions={len(regions)}, ckpt={len(results)}"
                )

    if pbar is not None:
        pbar.close()

    tmp = output_csv + ".tmp"
    pd.DataFrame(results).drop_duplicates(subset=["image"], keep="last").to_csv(tmp, index=False)
    os.replace(tmp, output_csv)
    print(f"[GPU {gpu_id}] done {len(done)}/{total_records}; saved {output_csv}", flush=True)


def detect_num_gpus():
    try:
        out = subprocess.check_output(["nvidia-smi", "-L"]).decode("utf-8").strip()
        return max(1, len([x for x in out.splitlines() if x.strip()]))
    except Exception:
        return max(1, torch.cuda.device_count())


mp.set_start_method("fork", force=True)
num_gpus = detect_num_gpus()
print("GPUs:", num_gpus, flush=True)

chunk_size = math.ceil(len(test_records) / num_gpus)
chunks = [test_records[i * chunk_size:(i + 1) * chunk_size] for i in range(num_gpus)]

processes = []
partials = []
for gpu_id, chunk in enumerate(chunks):
    if not chunk:
        continue
    output_csv = f"{HYBRID_PARTIAL_PREFIX}{gpu_id}.csv"
    partials.append(output_csv)
    p = mp.Process(target=worker_process, args=(gpu_id, chunk, output_csv))
    p.start()
    processes.append(p)

for p in processes:
    p.join()

bad_exitcodes = [p.exitcode for p in processes if p.exitcode not in (0, None)]
if bad_exitcodes:
    raise RuntimeError(f"One or more workers failed with exit codes: {bad_exitcodes}")

frames = []
for path in partials:
    if Path(path).exists():
        frame = pd.read_csv(path)
        print(f"Partial {path}: rows={len(frame)}", flush=True)
        frames.append(frame)
if not frames:
    raise RuntimeError("No partial outputs were created.")

final = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["image"], keep="last")
order = [Path(r["file_name"]).name for r in test_records]
final = final.set_index("image").reindex(order).reset_index()
final["regions"] = final["regions"].fillna("[]")
final.to_csv(OUTPUT_CSV, index=False)
print("Wrote", OUTPUT_CSV, "rows=", len(final), flush=True)
final.head()


In [ ]:
df = pd.read_csv(OUTPUT_CSV)
assert list(df.columns) == ["image", "regions"], df.columns
assert len(df) == len(test_records), (len(df), len(test_records))

bad = []
region_counts = []
for row in df.itertuples(index=False):
    try:
        parsed = json.loads(row.regions)
        assert isinstance(parsed, list)
        region_counts.append(len(parsed))
        for item in parsed:
            assert "bbox" in item and "type" in item and "text" in item
            assert isinstance(item["bbox"], list) and len(item["bbox"]) == 4
            assert item["type"] in VALID_TYPES
    except Exception as e:
        bad.append((row.image, str(e)))
        if len(bad) >= 5:
            break

print("Bad rows:", bad[:5])
print("Images:", len(df))
print("Total regions:", sum(region_counts))
print("Avg regions/page:", round(sum(region_counts) / max(1, len(region_counts)), 2))
print("Ready:", OUTPUT_CSV)

if RUN_SPLIT == "validation":
    solution_rows = []
    for rec in test_records:
        solution_rows.append({
            "image": Path(rec["file_name"]).name,
            "regions": json.dumps(rec.get("regions") or [], ensure_ascii=False),
        })
    solution_df = pd.DataFrame(solution_rows)
    try:
        from kaggle_metric import score_detailed

        breakdown = score_detailed(solution_df, df, "image")
        print("Local validation breakdown:", breakdown)
    except Exception as e:
        print("Skipped local metric. Add kaggle_metric.py from the official metric notebook to score locally.")
        print("Reason:", e)
